# Camada Gold — Painel Econômico Consolidado

Este notebook lê os 3 indicadores tratados (silver), agrega SELIC e Dólar
para média mensal (para ficarem na mesma granularidade do IPCA, que é
mensal), une as 3 séries numa única tabela por mês (JOIN), e salva o
resultado consolidado no S3, pronto para consulta e visualização.

In [0]:
dbutils.widgets.text("access_key", "")
dbutils.widgets.text("secret_key", "")

In [0]:
access_key = dbutils.widgets.get("access_key")
secret_key = dbutils.widgets.get("secret_key")

import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="sa-east-1"
)

## Leitura da camada Silver
Busca os 3 arquivos tratados (formato Parquet) salvos na pasta do dia atual.

In [0]:
import pandas as pd
import io
from datetime import datetime

data_hoje = datetime.today().strftime("%Y-%m-%d")
nomes = ["selic", "ipca", "dolar"]

dados_silver = {}

for nome in nomes:
    caminho = f"silver/{data_hoje}/{nome}.parquet"
    resposta = s3.get_object(Bucket="pipeline-economico-hugoqueiroz", Key=caminho)
    conteudo = resposta["Body"].read()
    dados_silver[nome] = pd.read_parquet(io.BytesIO(conteudo))
    print(f"{nome}: {dados_silver[nome].shape[0]} linhas")

selic: 679 linhas
ipca: 32 linhas
dolar: 679 linhas


## Agregação e JOIN via SQL
Registra cada indicador tratado como view temporária e usa SQL para
agregar SELIC e Dólar para média mensal, unindo as três séries em uma
única tabela consolidada por mês.

In [0]:
for nome, df in dados_silver.items():
    spark_df = spark.createDataFrame(df)
    spark_df.createOrReplaceTempView(f"{nome}_view")
    print(f"View criada: {nome}_view")

View criada: selic_view
View criada: ipca_view
View criada: dolar_view


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW painel_view AS
WITH selic_mensal AS (
  SELECT date_format(data, 'yyyy-MM') AS mes, 
  AVG(valor) AS selic_media
  FROM selic_view
  GROUP BY date_format(data, 'yyyy-MM')
),
dolar_mensal AS (
  SELECT date_format(data, 'yyyy-MM') AS mes, 
  AVG(valor) AS dolar_medio
  FROM dolar_view
  GROUP BY date_format(data, 'yyyy-MM')
),
ipca_mensal AS (
  SELECT date_format(data, 'yyyy-MM') AS mes, 
  valor AS ipca
  FROM ipca_view
)
SELECT s.mes, s.selic_media, d.dolar_medio, i.ipca
FROM selic_mensal s
INNER JOIN dolar_mensal d ON s.mes = d.mes
INNER JOIN ipca_mensal i ON s.mes = i.mes

In [0]:
%sql
SELECT * FROM painel_view ORDER BY mes  LIMIT 12

mes,selic_media,dolar_medio,ipca
2024-01,0.04373899999999998,4.914395454545455,0.42
2024-02,0.041957000000000015,4.964389473684211,0.83
2024-03,0.04142030000000001,4.980135,0.16
2024-04,0.04016799999999999,5.129095454545455,0.38
2024-05,0.03948380952380954,5.133047619047619,0.46
2024-06,0.03927000000000001,5.388974999999999,0.21
2024-07,0.03927000000000002,5.5420478260869555,0.38
2024-08,0.03927000000000002,5.552613636363636,-0.02
2024-09,0.03961209523809524,5.541566666666666,0.44
2024-10,0.04016799999999999,5.624108695652174,0.56


In [0]:
painel = spark.sql("SELECT * FROM painel_view ORDER BY mes").toPandas()
print(painel.head())
print(f"\nTotal: {painel.shape[0]} meses consolidados")

       mes  selic_media  dolar_medio  ipca
0  2024-01     0.043739     4.914395  0.42
1  2024-02     0.041957     4.964389  0.83
2  2024-03     0.041420     4.980135  0.16
3  2024-04     0.040168     5.129095  0.38
4  2024-05     0.039484     5.133048  0.46

Total: 32 meses consolidados


## Salvamento da camada Gold
Salva o painel consolidado (SELIC, IPCA e Dólar por mês) no S3, em formato
Parquet, pronto para consulta e análise.

In [0]:
import io

# Remove metadados internos do Databricks que impedem a serialização em parquet
painel_limpo = pd.DataFrame(painel.values, columns=painel.columns)

buffer = io.BytesIO()
painel_limpo.to_parquet(buffer, index=False)

caminho = f"gold/{data_hoje}/painel_economico.parquet"
s3.put_object(
    Bucket="pipeline-economico-hugoqueiroz",
    Key=caminho,
    Body=buffer.getvalue()
)

print(f"Salvo: {caminho}")

Salvo: gold/2026-09-15/painel_economico.parquet
